## Installs

In [141]:
# # Installs
# !pip install -qU "requests>=2.32.5" langchain-community pypdf faiss-cpu tiktoken transformers accelerate gradio "langchain>=0.2.0"  "sentence-transformers>=2.2.2" torch
# !pip install -qU "openai>=1.40.0" "python-dotenv>=1.0.1" langchain-openai

## Imports

In [142]:
from __future__ import annotations

# Standard library
import hashlib
import io
import os
import re
import json
import traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

# Third-party libraries
import gradio as gr
import requests
import torch
from pypdf import PdfReader
# from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# LangChain
from langchain.memory import ConversationBufferMemory
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()  # loads OPENAI_API_KEY from .env

# NEW: chain-based LLM imports
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable


In [ ]:
from langsmith import traceable
from langsmith.run_helpers import get_current_run_tree

@traceable
def my_child_function():
    current_run = get_current_run_tree()
    if current_run.parent_run_id:
        return current_run.parent_run_id
    else:
        print("This is a root run and has no parent.")

@traceable
def my_parent_function():
    return my_child_function()

# Call the parent function to initiate the trace
x = my_parent_function()
x

Parent Run ID: aea5e632-d976-402b-8b6f-c1e868bc74c0
aea5e632-d976-402b-8b6f-c1e868bc74c0
Parent Run ID: aea5e632-d976-402b-8b6f-c1e868bc74c0


UUID('aea5e632-d976-402b-8b6f-c1e868bc74c0')

## Utility Functions

In [ ]:


# Optional LangSmith bits
from langsmith import Client


# -----------------------------
# URL processing & downloading
# -----------------------------
def process_book_url(url: str) -> str:
    """
    Normalize a book URL so it's directly downloadable as a PDF.
    - Converts GitHub 'blob' URLs to 'raw' URLs.
    - Returns the original URL if no changes are needed.
    """
    # GitHub "blob" URL -> "raw" URL
    gh_blob = re.match(r"https?://github\.com/([^/]+)/([^/]+)/blob/(.+)", url)
    if gh_blob:
        user, repo, path = gh_blob.groups()
        return f"https://raw.githubusercontent.com/{user}/{repo}/{path}"

    # GitHub "tree" URLs sometimes appear; not a direct file
    gh_tree = re.match(r"https?://github\.com/([^/]+)/([^/]+)/tree/(.+)", url)
    if gh_tree:
        raise ValueError(
            "GitHub 'tree' URL provided. Please point to a specific file (PDF) or use a 'blob' link."
        )

    # Google Drive 'view' links, Dropbox share links, etc. could be added here if needed.
    return url


def download_pdf(
    url: str,
    save_path: Optional[str] = None,
    timeout: int = 60,
    verify_ssl: bool = True,
) -> Tuple[bytes, Optional[str]]:
    """
    Download a PDF from a (processed) URL.
    Returns (pdf_bytes, saved_path).
    - If save_path is provided, writes the PDF to disk and returns that path.
    - Raises requests.HTTPError for non-200 responses.
    """
    processed = process_book_url(url)
    resp = requests.get(processed, stream=True, timeout=timeout, verify=verify_ssl)
    resp.raise_for_status()

    # Read into memory
    pdf_bytes = resp.content

    # Optionally save to disk
    saved = None
    if save_path:
        # Ensure .pdf extension if missing
        if not save_path.lower().endswith(".pdf"):
            save_path = save_path + ".pdf"
        os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
        with open(save_path, "wb") as f:
            f.write(pdf_bytes)
        saved = save_path

    return pdf_bytes, saved


# --------------------------------
# PDF reading & page text extraction
# --------------------------------
def _open_pdf(pdf_source: Union[str, bytes, io.BytesIO]) -> PdfReader:
    """
    Internal helper to open a PDF from a path or bytes-like object.
    """
    if isinstance(pdf_source, str):
        # Path to file on disk
        with open(pdf_source, "rb") as f:
            return PdfReader(f)
    elif isinstance(pdf_source, bytes):
        return PdfReader(io.BytesIO(pdf_source))
    elif isinstance(pdf_source, io.BytesIO):
        return PdfReader(pdf_source)
    else:
        raise TypeError("pdf_source must be a file path (str), bytes, or BytesIO.")


def read_pdf_pages(
    pdf_source: Union[str, bytes, io.BytesIO],
    start_page: Optional[int] = None,
    end_page: Optional[int] = None,
) -> List[Dict[str, Union[int, str]]]:
    """
    Read a PDF and return a list of dicts: [{"page_number": int, "text": str}, ...]
    - page_number is 1-based (useful for human-readable citations).
    - start_page / end_page are 1-based inclusive bounds. If None, read the whole file.
    - Extracted text may be empty ('') for image-only pages; handle gracefully.
    """
    reader = _open_pdf(pdf_source)
    n = len(reader.pages)

    sp = 1 if start_page is None else max(1, start_page)
    ep = n if end_page is None else min(end_page, n)
    if sp > ep:
        return []

    pages = []
    for i in range(sp - 1, ep):  # convert to 0-based
        page = reader.pages[i]
        text = page.extract_text() or ""  # fall back to empty string
        # Normalize whitespace a bit (optional)
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        pages.append({"page_number": i + 1, "text": text.strip()})

    return pages


def get_pdf_page_numbers(pdf_source: Union[str, bytes, io.BytesIO]) -> List[int]:
    """
    Return a simple list of page numbers [1, 2, ..., N].
    Useful when you want to attach page metadata during vector indexing.
    """
    reader = _open_pdf(pdf_source)
    return list(range(1, len(reader.pages) + 1))



def stable_pdf_id(pdf_bytes: bytes) -> str:
    """
    Deterministic ID for the PDF content, handy for caching or reusing vector indexes.
    """
    return hashlib.sha256(pdf_bytes).hexdigest()[:16]


def _safe_json_extract(text: str) -> Dict[str, Any]:
    """Extract a JSON object from a model reply; fall back sensibly."""
    text = (text or "").strip()
    # common case: already a JSON object
    if text.startswith("{") and text.endswith("}"):
        try:
            return json.loads(text)
        except Exception:
            pass
    # lenient bracketed capture
    start, end = text.find("{"), text.rfind("}")
    if 0 <= start < end:
        try:
            return json.loads(text[start:end+1])
        except Exception:
            pass
    # ultimate fallback
    return {"score": 0.0, "rationale": text[:200]}

_DOC_REL_PROMPT = """
You are grading how relevant a context excerpt is to a user query.
Return a JSON object with fields:
- score: a float in [0,1] where 1.0 = perfectly relevant, 0.0 = unrelated
- rationale: a brief reason

User Query:
{query}

Context Excerpt:
{context}

Rules:
- Focus on topical relevance and utility for answering the query.
- Be strict about unrelated/tenuous content.
- Output ONLY JSON.
"""

_ANS_REL_PROMPT = """
You are grading how well an answer addresses a user query.
Return a JSON object with fields:
- score: a float in [0,1] where 1.0 = fully answers correctly and directly; 0.0 = fails to answer
- rationale: a brief reason

User Query:
{query}

Answer:
{answer}

Rules:
- Consider correctness, completeness, and directness.
- Ignore style/formatting quirks.
- Output ONLY JSON.
"""

@dataclass
class RAGEvalResult:
    per_doc: List[Dict[str, Any]]   # [{id, score, rationale}]
    avg_context_relevance: float
    answer_relevance: float

def evaluate_rag_run(
    query: str,
    retrieved_docs: List[Any],
    final_answer: str,
    llm_manager,  # your LLMManager instance (already loaded)
    *,
    langsmith_client: Optional[Client] = None,
    parent_run_id: Optional[str] = None,
    feedback_prefix: str = "rag_eval"
    ) -> RAGEvalResult:
    """
    Evaluate (1) per-document context relevance and (2) final answer relevance.
    All scores are in [0,1]. Uses your chain-based LLM via `llm_manager.generate(prompt)`.

    If `langsmith_client` and `parent_run_id` are provided, logs feedback metrics to LangSmith.
    """
    # 1) Per-document relevance
    per_doc: List[Dict[str, Any]] = []
    scores = []

    for i, doc in enumerate(retrieved_docs):
        # Accepts LangChain Document, dict, or plain string
        if hasattr(doc, "page_content"):
            content = getattr(doc, "page_content", "")
            meta = getattr(doc, "metadata", {}) or {}
        elif isinstance(doc, dict):
            content = doc.get("page_content") or doc.get("content") or doc.get("text") or ""
            meta = doc.get("metadata", {}) or {}
        else:
            content = str(doc)
            meta = {}

        label = (
            meta.get("id") or meta.get("source") or meta.get("path") or meta.get("url") or f"doc_{i}"
        )

        prompt = _DOC_REL_PROMPT.format(query=query, context=content[:4000])
        raw = llm_manager.generate(prompt)  # uses the chain-backed path
        parsed = _safe_json_extract(raw)
        score = parsed.get("score", 0.0)
        try:
            score = float(score)
        except Exception:
            score = 0.0
        score = max(0.0, min(1.0, score))

        per_doc.append({
            "id": label,
            "score": score,
            "rationale": parsed.get("rationale", ""),
        })
        scores.append(score)

        # Optional: push per-doc metric to LangSmith
        if langsmith_client and parent_run_id:
            try:
                langsmith_client.create_feedback(
                    run_id=parent_run_id,
                    key=f"{feedback_prefix}.ctx_relevance::{label}",
                    score=score,
                )
            except Exception:
                pass

    avg_ctx = sum(scores) / max(1, len(scores))

    # 2) Final answer relevance
    ans_prompt = _ANS_REL_PROMPT.format(query=query, answer=(final_answer or "")[:4000])
    ans_raw = llm_manager.generate(ans_prompt)
    ans_parsed = _safe_json_extract(ans_raw)
    ans_score = ans_parsed.get("score", 0.0)
    try:
        ans_score = float(ans_score)
    except Exception:
        ans_score = 0.0
    ans_score = max(0.0, min(1.0, ans_score))

    # Optional: push aggregate metrics to LangSmith
    print(f'Langsmith Client: {langsmith_client}, Parent Run ID: {parent_run_id}')
    
    if langsmith_client and parent_run_id:
        try:
            langsmith_client.create_feedback(
                run_id=parent_run_id,
                key=f"{feedback_prefix}.avg_context_relevance",
                score=avg_ctx,
            )
            langsmith_client.create_feedback(
                run_id=parent_run_id,
                key=f"{feedback_prefix}.answer_relevance",
                score=ans_score,
            )
            print(f"Logged RAG eval feedback to LangSmith (run {parent_run_id})")
        except Exception:
            pass

    return RAGEvalResult(per_doc=per_doc, avg_context_relevance=avg_ctx, answer_relevance=ans_score)

## RAG Components

In [145]:

    
@dataclass
class ChunkingConfig:
    chunk_size: int = 900
    chunk_overlap: int = 150
    separators: Optional[List[str]] = None  # override if you want custom split rules


class TutorAIVectorIndex:
    """
    Minimal vector index wrapper for RAG over a single book.
    - Chunking via RecursiveCharacterTextSplitter
    - Embeddings via sentence-transformers/all-MiniLM-L6-v2 (normalized)
    - Vector store: FAISS
    """
    def __init__(
        self,
        chunk_cfg: ChunkingConfig = ChunkingConfig(),
        embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2",
        device: Optional[str] = None,  # e.g., "cuda" or "cpu"
    ):
        # Handle common misspelling "allminilm-v6"
        if embedding_model.lower().replace("-", "").replace("_", "") in {
            "allminilmv6", "allminilml6v2"
        }:
            embedding_model = "sentence-transformers/all-MiniLM-L6-v2"

        # Text splitter
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_cfg.chunk_size,
            chunk_overlap=chunk_cfg.chunk_overlap,
            separators=chunk_cfg.separators,  # if None, uses sensible defaults
            length_function=len,
        )

        # Embeddings (normalized → cosine-friendly)
        self.embeddings = HuggingFaceEmbeddings(
            model_name=embedding_model,
            model_kwargs={"device": device} if device else {},
            encode_kwargs={"normalize_embeddings": True},
        )

        self.vs: Optional[FAISS] = None
        self._book_fingerprint: Optional[str] = None

    # 1) Generate chunks using RecursiveCharacterTextSplitter
    def make_chunks(
        self,
        pages: List[Dict[str, Any]],
        source_id: Optional[str] = None,   # e.g., stable_pdf_id from your utils
    ) -> List[Document]:
        """
        Input: pages = [{"page_number": int, "text": str}, ...] from your utility.
        Output: List[Document] with preserved page_number for citations.
        """
        docs: List[Document] = []
        for p in pages:
            text = (p.get("text") or "").strip()
            if not text:
                continue
            page_no = int(p.get("page_number", 0))
            # One doc per page before splitting
            docs.append(
                Document(
                    page_content=text,
                    metadata={
                        "page_number": page_no,
                        "source_id": source_id,
                    },
                )
            )

        # Split into chunks (preserving metadata on each split)
        chunk_docs = self.splitter.split_documents(docs)

        # Add a stable chunk id (handy later for tracing)
        for i, d in enumerate(chunk_docs):
            page_no = d.metadata.get("page_number", 0)
            base = f"{source_id or 'book'}|p{page_no}|{i}|{hashlib.md5(d.page_content.encode('utf-8')).hexdigest()[:8]}"
            d.metadata["chunk_id"] = base

        return chunk_docs

    # 2) Index chunks with FAISS (all-MiniLM-L6-v2, normalized)
    def build_index(self, chunk_docs: List[Document]) -> None:
        """
        Build/replace the index from chunk documents.
        """
        if not chunk_docs:
            raise ValueError("No chunk documents supplied to build_index().")
        self.vs = FAISS.from_documents(chunk_docs, self.embeddings)

        # Keep a tiny fingerprint to detect mismatches later (optional)
        fp_src = {d.metadata.get("source_id") for d in chunk_docs if d.metadata.get("source_id")}
        self._book_fingerprint = next(iter(fp_src)) if fp_src else None

    # 3) Retrieve top-k chunks for a query
    def retrieve(
        self,
        query: str,
        k: int = 5,
        with_scores: bool = True,
    ) -> List[Dict[str, Any]]:
        """
        Returns a list of dicts with: text, page_number, score (lower is better if using distances),
        and metadata (including chunk_id and source_id).
        """
        if self.vs is None:
            raise RuntimeError("Index not built yet. Call build_index() first.")

        # similarity_search_with_score returns tuples: (Document, score)
        # With normalized embeddings, score approximates cosine distance.
        pairs: List[Tuple[Document, float]] = self.vs.similarity_search_with_score(query, k=k)

        results: List[Dict[str, Any]] = []
        for doc, score in pairs:
            results.append({
                "text": doc.page_content,
                "page_number": doc.metadata.get("page_number"),
                "score": float(score) if with_scores else None,
                "metadata": doc.metadata,
                "citation": f"(p. {doc.metadata.get('page_number')})",
            })
        return results

    # Optional: persist & load (handy in Colab runs)
    def save(self, folder_path: str) -> None:
        """
        Save FAISS index + docstore to disk.
        """
        if self.vs is None:
            raise RuntimeError("Index not built yet. Nothing to save.")
        self.vs.save_local(folder_path)

    def load(self, folder_path: str) -> None:
        """
        Load FAISS index + docstore from disk.
        """
        self.vs = FAISS.load_local(folder_path, self.embeddings, allow_dangerous_deserialization=True)



## Conversation Pipeline

In [146]:
_CIT_RE = re.compile(r"\(p\.\s*(\d+)(?:\s*[-–]\s*\d+)?\)")
CITATION_REGEX = re.compile(r"\(p\.\s*\d+(?:\s*[-–]\s*\d+)?\)")  # (p. N) or (p. N–M)

# --- add near the top of your conv file ---
import re
from enum import Enum

class QueryIntent(str, Enum):
    BOOK_QA = "book_qa"
    CHITCHAT = "chitchat"

_CHITCHAT_PATTERNS = [
    r"^hi[!.]?$", r"^hello[!.]?$", r"^hey[!.]?$",
    r"\bthank(s| you)\b",
    r"\bhow (are|r) (you|u)\b",
    r"\bwhat can (you|u) do\b",
    r"\bwho are you\b", r"\bhelp\b", r"\bcapabilit(y|ies)\b",
    r"\btest\b", r"\bping\b",
]

def classify_intent(text: str) -> QueryIntent:
    t = (text or "").strip().lower()
    for pat in _CHITCHAT_PATTERNS:
        if re.search(pat, t):
            return QueryIntent.CHITCHAT
    # default to book QA
    return QueryIntent.BOOK_QA


# -----------------------------
# LLM registry & loader
# -----------------------------
@dataclass
class LLMConfig:
    repo_id: str
    is_chat: bool = True              # affects prompting slightly
    trust_remote_code: bool = False   # set True for models that require it
    dtype: Optional[str] = None       # "bfloat16" | "float16" | "float32"
    device_map: Optional[str] = "auto"


DEFAULT_LLM_CATALOG: Dict[str, LLMConfig] = {
    # Keep this list small & flexible for the POC; add more as you like.
    # Pick models you actually have resources to run on Colab.
    "mistral-7b-instruct": LLMConfig(
        repo_id="mistralai/Mistral-7B-Instruct-v0.3", dtype="bfloat16"
    ),
    "llama-3-8b-instruct": LLMConfig(
        repo_id="meta-llama/Meta-Llama-3-8B-Instruct", dtype="bfloat16"
    ),
    "gemma-2-2b-it": LLMConfig(
        repo_id="google/gemma-2-2b-it", dtype="bfloat16"
    ),
    "phi-3-mini-4k-instruct": LLMConfig(
        repo_id="microsoft/Phi-3-mini-4k-instruct", dtype="bfloat16"
    ),
    "Qwen2.5-0.5B-Instruct": LLMConfig(
        repo_id="Qwen/Qwen2.5-0.5B-Instruct", dtype="bfloat16"
    ),
}


@dataclass
class LLMConfig:
    provider: str = 'openai'  # 'hf' or 'openai'
    repo_id: str | None = None
    is_chat: bool = True
    trust_remote_code: bool = False
    dtype: Optional[str] = None
    device_map: Optional[str] = "auto"


DEFAULT_LLM_CATALOG: Dict[str, LLMConfig] = {
    # OpenAI hosted
    "gpt-4o-mini": LLMConfig(provider="openai"),
    "gpt-4o": LLMConfig(provider="openai"),
    # Sample local HF models (left as examples; keep if you want to use them)
    "mistral-7b-instruct": LLMConfig(repo_id="mistralai/Mistral-7B-Instruct-v0.3", dtype="bfloat16"),
    "llama-3-8b-instruct": LLMConfig(repo_id="meta-llama/Meta-Llama-3-8B-Instruct", dtype="bfloat16"),
    "gemma-2-2b-it": LLMConfig(repo_id="google/gemma-2-2b-it", dtype="bfloat16"),
    "phi-3-mini-4k-instruct": LLMConfig(repo_id="microsoft/Phi-3-mini-4k-instruct", dtype="bfloat16"),
    "Qwen2.5-0.5B-Instruct": LLMConfig(repo_id="Qwen/Qwen2.5-0.5B-Instruct", dtype="bfloat16"),
}


class LLMManager:
    """Load and run LLMs (OpenAI via API or local HF via transformers)."""
    def __init__(self, catalog: Optional[Dict[str, LLMConfig]] = None):
        self.catalog = catalog or DEFAULT_LLM_CATALOG
        self.pipe = None  # for HF pipeline
        self.client: Optional[OpenAI] = None  # for OpenAI
        self.model_name: Optional[str] = None
        self.provider: Optional[str] = None
        self.gen_defaults: Dict[str, Any] = {}

        # NEW: chain model holder (kept None unless OpenAI provider is loaded)
        self.lcel_model: Optional[ChatOpenAI] = None

    def load(self, name: str, gen_kwargs: Optional[Dict[str, Any]] = None):
        if name not in self.catalog:
            raise ValueError(f"Unknown model '{name}'. Available: {list(self.catalog)}")
        cfg = self.catalog[name]
        self.model_name = name
        self.provider = cfg.provider

        if cfg.provider == "openai":
            # Uses OPENAI_API_KEY from env (dotenv already loaded at import time)
            self.client = OpenAI()
            self.pipe = None
            self.gen_defaults = {
                "temperature": 0.2,
                "top_p": 0.9,
                "max_tokens": 512,
                **(gen_kwargs or {}),
            }

            # NEW: prepare a chain-capable chat model (reads OPENAI_API_KEY from env)
            self.lcel_model = ChatOpenAI(model=self.model_name)

        else:
            # HuggingFace transformers local pipeline (kept for completeness)
            from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline as hf_pipeline

            dtype = {
                "bfloat16": torch.bfloat16,
                "float16": torch.float16,
                "float32": torch.float32,
                None: None,
            }[cfg.dtype]

            tok = AutoTokenizer.from_pretrained(cfg.repo_id, trust_remote_code=cfg.trust_remote_code)
            model = AutoModelForCausalLM.from_pretrained(
                cfg.repo_id,
                torch_dtype=dtype,
                device_map=cfg.device_map,
                trust_remote_code=cfg.trust_remote_code,
            )
            self.pipe = hf_pipeline(
                "text-generation",
                model=model,
                tokenizer=tok,
                return_full_text=False,
                pad_token_id=tok.eos_token_id,
            )
            self.client = None
            self.gen_defaults = {
                "max_new_tokens": 512,
                "temperature": 0.2,
                "top_p": 0.9,
                "do_sample": True,
                **(gen_kwargs or {}),
            }

    def generate(self, prompt: str, **overrides) -> str:
        if self.provider == "openai":
            if self.lcel_model is None:
                raise RuntimeError("Chat model not initialized. Call load(name) first.")
            params = {**self.gen_defaults, **overrides}
            # translate common knobs where relevant
            temperature = params.get("temperature", 0.2)
            top_p = params.get("top_p", 0.9)
            max_tokens = params.get("max_tokens", 512)

            # chain: ChatOpenAI -> string
            chain = self.lcel_model.bind(
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_tokens,
            ) | StrOutputParser()

            return chain.invoke(prompt).strip()

        else:
            if self.pipe is None:
                raise RuntimeError("HF LLM not loaded. Call load(name) first.")
            out = self.pipe(prompt, **{**self.gen_defaults, **overrides})
            return out[0]["generated_text"]

# -----------------------------
# Prompt template (with placeholders)
# -----------------------------
SYSTEM_TEMPLATE = """\
You are TutorAI, an AI tutor that answers STRICTLY from the provided book context.
Rules:
- Only use the context (extracted from the book) to answer.
- If the answer is not in the context, politely refuse and say it's not in the text.
- Always include page citations in the form (p. <page_number>) for each claim.
- Be concise but clear. Use bullet points where appropriate.
- Maintain continuity with conversation history when helpful.
{system_description}
"""

GUIDELINES_TEMPLATE = """\
Guidelines:
{guidelines}
"""

# ChatPromptTemplate supports message roles + placeholders
CONV_PROMPT = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_TEMPLATE),
        MessagesPlaceholder("history"),
        (
            "human",
            # context is injected separately (retrieved chunks)
            "User query:\n{user_query}\n\n"
            "Book context (use ONLY this; provide citations for each relevant statement):\n{context}\n\n"
            + GUIDELINES_TEMPLATE
        ),
    ]
)


# -----------------------------
# Conversation pipeline
# -----------------------------
@dataclass
class ConversationConfig:
    top_k: int = 5
    max_context_chars: int = 6000         # guardrail to avoid exceeding model limits
    min_chars_per_chunk: int = 120        # filter very tiny chunks
    join_delimiter: str = "\n\n---\n\n"   # separator between chunks

@dataclass
class GuardrailConfig:
    mode: str = "append_once"   # "append_once" | "per_paragraph"
    require_on_any_output: bool = True
    paragraph_min_chars: int = 20          # ignore tiny paras when per_paragraph
    max_auto_citations_per_answer: int = 1 # for append_once
    refusal_text: str = (
        "I can’t find that in the provided text. Please point me to a specific section, "
        "or ask about content that appears in the book."
    )

class TutorAIConversationPipeline:
    def __init__(
        self,
        indexer,  # instance of TutorAIVectorIndex
        llm_manager: Optional[LLMManager] = None,
        conv_cfg: ConversationConfig = ConversationConfig(),
        guard_cfg: GuardrailConfig = GuardrailConfig(),
    ):
        self.indexer = indexer
        self.llm = llm_manager or LLMManager()
        self.cfg = conv_cfg
        self.guard = guard_cfg

        from langchain.memory import ConversationBufferMemory
        self.memory = ConversationBufferMemory(
            return_messages=True, memory_key="history", input_key="user_query"
        )

    def read_user_query(self, text: str) -> str:
        return (text or "").strip()

    def _retrieve_context_blocks(self, query: str) -> List[Dict[str, Any]]:
        hits = self.indexer.retrieve(query, k=self.cfg.top_k)
        cleaned = [h for h in hits if len(h["text"]) >= self.cfg.min_chars_per_chunk]
        return cleaned or hits

    def _format_context(self, blocks: List[Dict[str, Any]]) -> str:
        parts, running_len = [], 0
        for h in blocks:
            txt = h["text"].strip()
            cite = f"(p. {h.get('page_number')})"
            chunk = f"[Page {h.get('page_number')}]\n{txt}\n{cite}"
            if running_len + len(chunk) > self.cfg.max_context_chars:
                break
            parts.append(chunk)
            running_len += len(chunk) + len(self.cfg.join_delimiter)
        return self.cfg.join_delimiter.join(parts)

    def _render_prompt(
        self,
        user_query: str,
        context: str,
        system_description: str = "",
        guidelines: str = "Answer only from the context and include page citations.",
    ) -> str:
        from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
        from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

        SYSTEM_TEMPLATE = """You are TutorAI, an AI tutor that answers STRICTLY from the provided book context.
Rules:
- Only use the context (extracted from the book) to answer.
- If the answer is not in the context, politely refuse and say it's not in the text.
- Always include page citations in the form (p. <page_number>) for each claim.
- Be concise but clear. Use bullet points where appropriate.
- Maintain continuity with conversation history when helpful.
{system_description}
"""
        GUIDELINES_TEMPLATE = "Guidelines:\n{guidelines}"
        CONV_PROMPT = ChatPromptTemplate.from_messages(
            [
                ("system", SYSTEM_TEMPLATE),
                MessagesPlaceholder("history"),
                (
                    "human",
                    "User query:\n{user_query}\n\n"
                    "Book context (use ONLY this; provide citations for each relevant statement):\n{context}\n\n"
                    + GUIDELINES_TEMPLATE
                ),
            ]
        )

        history = self.memory.load_memory_variables({}).get("history", [])
        msgs = CONV_PROMPT.format_messages(
            system_description=system_description,
            user_query=user_query,
            context=context if context.strip() else "(No relevant context found.)",
            guidelines=guidelines,
            history=history,
        )

        rendered = []
        for m in msgs:
            if isinstance(m, SystemMessage):
                rendered.append(f"<|system|>\n{m.content}\n")
            elif isinstance(m, HumanMessage):
                rendered.append(f"<|user|>\n{m.content}\n")
            else:
                rendered.append(f"<|assistant|>\n{m.content}\n")
        rendered.append("<|assistant|>\n")
        return "\n".join(rendered)

    # ---------- Guardrail verifier ----------
    def _extract_pages_from_blocks(self, blocks: List[Dict[str, Any]]) -> List[int]:
        pages = [b.get("page_number") for b in blocks if b.get("page_number")]
        # preserve order of appearance, dedupe
        seen, ordered = set(), []
        for p in pages:
            if p not in seen:
                ordered.append(int(p))
                seen.add(p)
        return ordered

    def _has_any_citation(self, text: str) -> bool:
        return bool(CITATION_REGEX.search(text))

    def _append_once_citation(self, answer: str, pages: List[int]) -> str:
        if not pages:
            return answer
        pages_str = ", ".join(str(p) for p in pages)
        suffix = f" (p. {pages_str})"
        # Avoid appending twice
        if not answer.rstrip().endswith(")"):
            return answer.rstrip() + " " + suffix
        return answer.rstrip() + " " + suffix

    def _per_paragraph_citations(self, answer: str, pages: List[int]) -> str:
        if not pages:
            return answer
        paras = [p for p in answer.split("\n")]

        # cycle through pages to distribute citations
        i = 0
        fixed = []
        for para in paras:
            clean = para.rstrip()
            if len(clean.strip()) < self.cfg.min_chars_per_chunk:
                fixed.append(clean)
                continue
            if CITATION_REGEX.search(clean):
                fixed.append(clean)
            else:
                page = pages[i % len(pages)]
                fixed.append(f"{clean} (p. {page})")
                i += 1
        return "\n".join(fixed)

    def _enforce_refusal_when_no_context(self, answer: str, blocks: List[Dict[str, Any]]) -> str:
        if blocks:
            return answer
        # No context → ensure refusal
        if self.guard.refusal_text.lower() not in answer.lower() and "(p." not in answer:
            return self.guard.refusal_text
        return answer

    def _guardrail_verify_and_fix(self, answer: str, blocks: List[Dict[str, Any]]) -> str:
        # First, if we had no retrieved context, enforce refusal
        answer = self._enforce_refusal_when_no_context(answer, blocks)

        pages = self._extract_pages_from_blocks(blocks)
        if not self.guard.require_on_any_output:
            return answer

        if self.guard.mode == "per_paragraph":
            # Ensure each paragraph has a citation
            return self._per_paragraph_citations(answer, pages)

        # append_once (default)
        if not self._has_any_citation(answer):
            # Cap the total appended groups to avoid spam
            return self._append_once_citation(answer, pages[: max(1, self.guard.max_auto_citations_per_answer)])
        return answer
    # ---------- /Guardrail verifier ----------

    def _guardrail_verify_and_fix(
        self, answer: str, blocks: List[Dict[str, Any]], require_citations: bool = True
    ) -> str:
        # If no citations required (like chit-chat), only enforce refusal
        if not require_citations:
            return self._enforce_refusal_when_no_context(answer, blocks)

        # Otherwise run the full citation guard
        answer = self._enforce_refusal_when_no_context(answer, blocks)
        pages = self._extract_pages_from_blocks(blocks)

        if not self.guard.require_on_any_output:
            return answer

        if self.guard.mode == "per_paragraph":
            return self._per_paragraph_citations(answer, pages)

        if not self._has_any_citation(answer):
            return self._append_once_citation(
                answer, pages[: max(1, self.guard.max_auto_citations_per_answer)]
            )
        return answer

    def _chitchat_reply(self, user_query: str) -> str:
        # keep it short; no book claims, no citations
        return (
            "I’m TutorAI. I can answer questions strictly from *Storytelling with Data* "
            "and cite pages. Ask about charts, storytelling, design principles, or any topic from the book."
        )


    def chat(
        self,
        user_query: str,
        *,
        llm_name: str,
        system_description: str = "",
        guidelines: str = "Answer only from the context and include page citations.",
        llm_overrides: Optional[Dict[str, Any]] = None,
        force_intent: Optional[QueryIntent] = None,   # allow override from UI if needed
    ) -> Dict[str, Any]:
        if self.llm.pipe is None or self.llm.model_name != llm_name:
            self.llm.load(llm_name)

        query = self.read_user_query(user_query)
        intent = force_intent or classify_intent(query)

        # --- CHITCHAT path: bypass RAG, no citations ---
        if intent == QueryIntent.CHITCHAT:
            answer = self._chitchat_reply(query)
            self.memory.chat_memory.add_user_message(query)
            self.memory.chat_memory.add_ai_message(answer)
            return {
                "model": llm_name,
                "answer": answer,
                "citations_pages": [],
                "retrieved_context": [],
                "history_len": len(self.memory.chat_memory.messages),
            }

        # --- BOOK_QA path: regular RAG ---
        blocks = self._retrieve_context_blocks(query)
        context = self._format_context(blocks)

        prompt = self._render_prompt(
            user_query=query,
            context=context if context.strip() else "(No relevant context found.)",
            system_description=system_description,
            guidelines=guidelines,
        )

        answer = self.llm.generate(prompt, **(llm_overrides or {})).strip()

        # require citations only for BOOK_QA
        answer = self._guardrail_verify_and_fix(answer, blocks, require_citations=True)

        self.memory.chat_memory.add_user_message(query)
        self.memory.chat_memory.add_ai_message(answer)

        pages = sorted({b["page_number"] for b in blocks if b.get("page_number")})
        return {
            "model": llm_name,
            "answer": answer,
            "citations_pages": pages,
            "retrieved_context": blocks,
            "history_len": len(self.memory.chat_memory.messages),
        }

    def _extract_pages_from_answer(self, text: str):
        pages = [int(m.group(1)) for m in _CIT_RE.finditer(text or "")]
        # dedupe, keep order
        seen, out = set(), []
        for p in pages:
            if p not in seen:
                out.append(p); seen.add(p)
        return out


## UI

In [147]:

class TutorAIDashboard:
    """
    Gradio dashboard wrapper with:
      - Chatbot
      - Text input
      - Model dropdown
      - Context (top-k) slider
      - Temperature slider
      - Similarity (max distance) slider
      - One-button initializer that:
          * downloads the PDF
          * reads pages
          * builds FAISS index
          * sets up the conversation pipeline
    """

    def __init__(
        self,
        default_book_url: str,
        default_model_key: Optional[str] = None,
        chunk_cfg: Optional[ChunkingConfig] = None,
        guard_mode: str = "append_once",  # or "per_paragraph"
        device_embeddings: Optional[str] = None,  # "cuda" for GPU embeddings
    ):
        self.default_book_url = default_book_url
        self.default_model_key = default_model_key  # if None, we'll use the first listed model
        self.llm_config = DEFAULT_LLM_CATALOG[self.default_model_key]
        self.chunk_cfg = chunk_cfg or ChunkingConfig(chunk_size=900, chunk_overlap=150)
        self.guard_mode = guard_mode
        self.device_embeddings = device_embeddings
        # Runtime objects created during initialization
        self.indexer: Optional[TutorAIVectorIndex] = None
        self.conv: Optional[TutorAIConversationPipeline] = None

        # Presentation strings
        self.default_system = "TutorAI for employees learning from 'Storytelling with Data'."
        self.default_guidelines = (
            "Answer ONLY from the provided context. Include (p. N) citation(s). "
            "If the information is not in the context, refuse."
        )

        # LLM catalog from LLMManager (so we can fill the dropdown even before init)
        self.llm_manager = LLMManager(self.llm_config)
        # self.model_choices = list(self.llm_manager.catalog.keys())
        # if self.default_model_key is None and self.model_choices:
        #     self.default_model_key = self.model_choices[0]

    # ------------------ Initialization orchestration ------------------
    def initialize_and_build(
        self,
        book_url: str,
        embedding_model: str,
        llm_key: str,
        progress=gr.Progress(track_tqdm=False),
    ) -> str:
        """
        Runs the end-to-end setup:
          1) Download & read PDF
          2) Chunk with LangChain splitter
          3) Build FAISS index
          4) Create conversation pipeline with guardrails
        Returns a short status string for the UI.
        """
        try:
            progress(0, desc="Processing book URL")
            processed_url = process_book_url(book_url.strip() or self.default_book_url)

            progress(0.15, desc="Downloading PDF")
            pdf_bytes, saved_path = download_pdf(processed_url, save_path="data/storytelling_with_data.pdf")
            pdf_id = stable_pdf_id(pdf_bytes)

            progress(0.35, desc="Reading pages from PDF")
            pages = read_pdf_pages(pdf_bytes)
            if not pages:
                return "No text could be extracted from the PDF. Consider adding OCR."

            progress(0.5, desc="Building indexer and chunking")
            self.indexer = TutorAIVectorIndex(
                chunk_cfg=self.chunk_cfg,
                embedding_model=embedding_model,
                device=self.device_embeddings,           # "cuda" if you want GPU embeddings
            )
            chunk_docs = self.indexer.make_chunks(pages, source_id=pdf_id)

            progress(0.7, desc="Constructing FAISS index")
            self.indexer.build_index(chunk_docs)

            progress(0.85, desc=f"Loading model: {llm_key}")
            # Build conversation pipeline with guardrails
            guard_cfg = GuardrailConfig(mode=self.guard_mode)
            self.conv = TutorAIConversationPipeline(
                indexer=self.indexer,
                guard_cfg=guard_cfg
            )
            # Optionally pre-load model here so first query is fast
            self.conv.llm.load(llm_key)

            progress(1.0, desc="Initialization complete")
            return f"Initialized successfully. Chunks: {len(chunk_docs)}. Model loaded: {llm_key}."
        except Exception as e:
            return "Initialization failed:\n" + "".join(traceback.format_exception_only(type(e), e)).strip()

    # ------------------ Chat engine with similarity cutoff ------------------
    def _ensure_model_loaded(self, model_key: str):
        if self.conv is None:
            raise RuntimeError("Pipeline not initialized yet. Click 'Initialize & Build' first.")
        if self.conv.llm.pipe is None or self.conv.llm.model_name != model_key:
            self.conv.llm.load(model_key)

    @traceable
    def _rag_chat_with_cutoff(
        self,
        user_query: str,
        llm_name: str,
        top_k: int,
        temperature: float,
        max_distance: float,
        system_description: str,
        guidelines: str,
        run_id: Optional[str] = None,
    ) -> Dict[str, Any]:
        if self.conv is None:
            raise RuntimeError("Pipeline not initialized yet. Click 'Initialize & Build' first.")

        # intent
        intent = classify_intent(user_query)

        self.conv.cfg.top_k = max(1, int(top_k))
        self.conv.llm.gen_defaults["temperature"] = float(temperature)

        if intent == QueryIntent.CHITCHAT:
            # bypass retrieval entirely; no citations
            return self.conv.chat(
                user_query, llm_name=llm_name,
                system_description=system_description, guidelines=guidelines,
                llm_overrides=None, force_intent=QueryIntent.CHITCHAT
            )

        # BOOK_QA: retrieve -> apply cutoff
        raw_blocks = self.conv.indexer.retrieve(user_query, k=self.conv.cfg.top_k)
        blocks = [b for b in raw_blocks if (b.get("score") is None or b["score"] <= max_distance)]
        # IMPORTANT: do NOT fallback to a random page; let the system refuse if nothing relevant
        if not blocks:
            # build prompt with empty context by calling chat() with no forced blocks.
            # We can temporarily monkey-patch a minimal method to feed empty blocks,
            # but simpler is to set top_k=0 here and let chat() re-retrieve.
            # Instead, we directly emulate its steps:
            context = "(No relevant context found.)"
            prompt = self.conv._render_prompt(
                user_query=user_query, context=context,
                system_description=system_description, guidelines=guidelines
            )
            # answer = self.conv.llm.generate(prompt).strip()
            response = self.conv.chat(user_query=user_query, llm_name=llm_name)
            answer = response["answer"]
            retrieved_context = response['retrieved_context']

            # self.current_run_id = my_parent_function()
            evaluate_rag_run(
                query=user_query, 
                retrieved_docs=retrieved_context,
                final_answer=answer, 
                llm_manager=self.conv.llm,
                langsmith_client=Client(),
                parent_run_id=run_id
                )

            # require refusal (no context) and no auto-citation
            answer = self.conv._guardrail_verify_and_fix(answer, [], require_citations=True)

            self.conv.memory.chat_memory.add_user_message(user_query)
            self.conv.memory.chat_memory.add_ai_message(answer)
            return {"model": llm_name, "answer": answer, "pages": []}

        # If we have blocks, proceed normally but **without** forcing extra pages
        context = self.conv._format_context(blocks)
        prompt = self.conv._render_prompt(
            user_query=user_query, context=context,
            system_description=system_description, guidelines=guidelines
        )
        answer = self.conv.llm.generate(prompt).strip()
        answer = self.conv._guardrail_verify_and_fix(answer, blocks, require_citations=True)

        self.conv.memory.chat_memory.add_user_message(user_query)
        self.conv.memory.chat_memory.add_ai_message(answer)
        pages = sorted({b["page_number"] for b in blocks if b.get("page_number")})
        return {"model": llm_name, "answer": answer, "pages": pages}


    # ------------------ Gradio wiring ------------------
    def _on_initialize_click(self, book_url, embedding_model, model_key):
        status = self.initialize_and_build(book_url, embedding_model, model_key)
        return status

    def _on_send(self, message, chat_history, model, top_k, temperature, cutoff):
        try:
            self._ensure_model_loaded(model)
            result = self._rag_chat_with_cutoff(
                user_query=message,
                llm_name=model,
                top_k=top_k,
                temperature=temperature,
                max_distance=cutoff,
                system_description=self.default_system,
                guidelines=self.default_guidelines,
            )
            chat_history = chat_history + [
                (message, f"{result['answer']}\n\nCitations: pages {result['pages']}")
            ]
            return "", chat_history
        except Exception as e:
            chat_history = chat_history + [(message, f"Error: {e}")]
            return "", chat_history

    def _on_clear(self):
        if self.conv is not None:
            self.conv.memory.clear()
        return []

    # Public: build the Blocks app
    def build_app(self):
        with gr.Blocks(theme=gr.themes.Soft(), fill_height=True) as demo:
            gr.Markdown("## TutorAI — RAG Dashboard")

            with gr.Row():
                with gr.Column(scale=3):
                    chatbot = gr.Chatbot(
                        label="TutorAI",
                        height=500,
                        show_copy_button=True,
                        bubble_full_width=False,
                    )
                    with gr.Row():
                        msg = gr.Textbox(
                            label="Your question",
                            placeholder="Ask about 'Storytelling with Data' (answers must cite p. N)",
                            scale=20,
                        )
                        send = gr.Button("Send", variant="primary", scale=1)
                    clear = gr.Button("Clear conversation")

                with gr.Column(scale=2):
                    gr.Markdown("### Initialization")
                    book_url = gr.Textbox(
                        label="Book URL",
                        value=self.default_book_url,
                        interactive=True,
                    )
                    embedding_model = gr.Textbox(
                        label="Embedding model (HF)",
                        value="sentence-transformers/all-MiniLM-L6-v2",
                        interactive=True,
                    )
                    model = gr.Dropdown(
                        label="Model",
                        choices=self.model_choices,
                        value=self.default_model_key,
                        interactive=True,
                    )
                    init_status = gr.Markdown(value="Not initialized.")
                    init_btn = gr.Button("Initialize & Build")

                    gr.Markdown("### Retrieval and Generation Controls")
                    top_k = gr.Slider(
                        minimum=1, maximum=12, value=5, step=1,
                        label="Context documents (top-k)"
                    )
                    temperature = gr.Slider(
                        minimum=0.0, maximum=1.5, value=0.2, step=0.05,
                        label="Temperature"
                    )
                    cutoff = gr.Slider(
                        minimum=0.0, maximum=1.0, value=0.6, step=0.01,
                        label="Similarity (max distance)",
                        info="Lower distance = more similar. Chunks above this distance are dropped.",
                    )

            # Events
            init_btn.click(
                fn=self._on_initialize_click,
                inputs=[book_url, embedding_model, model],
                outputs=[init_status],
            )

            send.click(
                fn=self._on_send,
                inputs=[msg, chatbot, model, top_k, temperature, cutoff],
                outputs=[msg, chatbot],
            )
            msg.submit(
                fn=self._on_send,
                inputs=[msg, chatbot, model, top_k, temperature, cutoff],
                outputs=[msg, chatbot],
            )
            clear.click(fn=self._on_clear, outputs=[chatbot])

        return demo

    # Optional helper to launch directly
    def launch(self, **kwargs):
        app = self.build_app()
        return app.launch(**kwargs)




In [148]:
DEFAULT_BOOK_URL = "https://github.com/infoalpha/Data-Science-books/blob/master/storytelling-with-data-cole-nussbaumer-knaflic.pdf"


dashboard = TutorAIDashboard(
    default_book_url=DEFAULT_BOOK_URL,
    default_model_key='gpt-4o-mini',        # choose at runtime
    chunk_cfg=ChunkingConfig(chunk_size=900, chunk_overlap=150),
    guard_mode="per_paragraph",    # or "append_once"
    # device_embeddings="cuda",      # use GPU for embeddings if available
)
# app = dashboard.build_app()
# app.launch(debug=True, show_error=True)

In [149]:
dashboard.initialize_and_build(
    book_url=DEFAULT_BOOK_URL,
    embedding_model="sentence-transformers/all-MiniLM-L6-v2",
    llm_key="gpt-4o-mini",)

'Initialized successfully. Chunks: 612. Model loaded: gpt-4o-mini.'

In [150]:
current_run_id = my_parent_function()
current_run_id

Parent Run ID: 8d3c8261-25c3-4ec8-990e-64d6640776e5
8d3c8261-25c3-4ec8-990e-64d6640776e5
Parent Run ID: 8d3c8261-25c3-4ec8-990e-64d6640776e5


UUID('8d3c8261-25c3-4ec8-990e-64d6640776e5')

In [ ]:
current_run_id = my_parent_function()
print("Current run ID:", current_run_id)
dashboard._rag_chat_with_cutoff(
    "What are the key principles of effective data visualization discussed in the book?",
    llm_name="gpt-4o-mini",
    top_k=5,
    temperature=0.7,
    max_distance=0.5,
    system_description=dashboard.default_system,
    guidelines=dashboard.default_guidelines,
    run_id=current_run_id
)

Parent Run ID: f3088402-447c-4af1-8ff1-a400dd8b29f2
f3088402-447c-4af1-8ff1-a400dd8b29f2
Parent Run ID: f3088402-447c-4af1-8ff1-a400dd8b29f2
Current run ID: f3088402-447c-4af1-8ff1-a400dd8b29f2
